# ArkClaw Python SDK Best Practices

This notebook shows a practical ArkClaw SDK workflow using the current `arkclaw` package and CLI-era API shape.

Covered topics:

- Configure the SDK client from environment variables.
- Use runtime options for timeout and retry overrides.
- List spaces and prepare required identifiers.
- Create users and instances.
- Query, wait for, and update instances.
- Get a chat token and send a message through a WebSocket session.
- Create and inspect command jobs.

Before running, export real credentials and resource identifiers, or edit the placeholder values in the setup cell.


## 0. Optional Environment Setup

If you run this notebook locally, set credentials in your shell before starting Jupyter:

```bash
export ARKCLAW_ACCESS_KEY="your-ak"
export ARKCLAW_SECRET_KEY="your-sk"
export ARKCLAW_REGION="cn-beijing"
```

The SDK also supports `VOLCENGINE_ACCESS_KEY`, `VOLCENGINE_SECRET_KEY`, and `VOLCENGINE_REGION`.


In [ ]:
import json
from datetime import datetime

from arkclaw import ApiError, ArkClawClient, RuntimeOptions, ValidationError


def print_json(title: str, data: object) -> None:
    print(f"\n## {title}")
    print(json.dumps(data, ensure_ascii=False, indent=2))


client = ArkClawClient.from_env(
    connect_timeout=10,
    read_timeout=30,
    max_retries=3,
    retry_backoff=0.5,
    max_retry_backoff=30,
)

runtime = RuntimeOptions(
    read_timeout=60,
    max_retries=5,
)

SPACE_ID = "csi-replace-me"
USER_ID = "user-replace-me"
INSTANCE_ID = "ci-replace-me"

timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
INSTANCE_NAME = f"sdk-notebook-{timestamp}"
COMMAND_JOB_NAME = f"sdk-notebook-job-{timestamp}"

print("SDK client is ready.")


## 1. Preflight Checks

Fill `SPACE_ID` and `USER_ID` before running create/update cells. `INSTANCE_ID` can either be filled manually or produced by the create-instance cell below.


In [ ]:
if SPACE_ID == "csi-replace-me":
    raise ValueError("Please set SPACE_ID before running this notebook.")

if USER_ID == "user-replace-me":
    raise ValueError("Please set USER_ID before creating an instance.")

print("SPACE_ID =", SPACE_ID)
print("USER_ID =", USER_ID)


## 2. List Spaces

Use `ListClawSpaces` to confirm account-level access and discover existing spaces.


In [ ]:
spaces = client.spaces.list(runtime_options=runtime)
print_json("ListClawSpaces", spaces)


## 3. Create Users in Batch

If your environment already has users, you can skip this cell and keep using the existing `USER_ID`.


In [ ]:
# Uncomment and fill real user information before running.
# create_users_result = client.users.create_many(
#     space_id=SPACE_ID,
#     users=[
#         {
#             "email": "alice@example.com",
#             "name": "Alice",
#             "preferred_username": "alice",
#         }
#     ],
#     runtime_options=runtime,
# )
# print_json("CreateUsers", create_users_result)


## 4. Create an Instance

Create an ArkClaw instance in the selected space. Use a timestamped name to avoid conflicts in repeated notebook runs.


In [ ]:
create_result = client.instances.create(
    space_id=SPACE_ID,
    user_id=USER_ID,
    instance_name=INSTANCE_NAME,
    seat_type="Standard",
    description="Created from arkclaw-python-sdk best-practice notebook",
    runtime_options=runtime,
)

INSTANCE_ID = create_result.get("InstanceId", INSTANCE_ID)
print_json("CreateClawInstance", create_result)
print("INSTANCE_ID =", INSTANCE_ID)


## 5. Query Instance Detail and List View

After creation, query both the single-instance detail and list view to verify visibility and status.


In [ ]:
detail = client.instances.get(
    space_id=SPACE_ID,
    instance_id=INSTANCE_ID,
    runtime_options=runtime,
)

instances = client.instances.list(
    space_id=SPACE_ID,
    max_results=20,
    runtime_options=runtime,
)

print_json("GetClawInstance", detail)
print_json("ListClawInstances", instances)


## 6. Wait for the Instance to Become Running

Instance creation is asynchronous. Prefer the workflow helper instead of hand-writing polling loops in application code.


In [ ]:
running = client.workflows.wait_for_instance(
    space_id=SPACE_ID,
    instance_id=INSTANCE_ID,
    target_status="Running",
    timeout=600,
    interval=5,
)
print_json("WaitForInstance", running)


## 7. Prepare Chat Access

`prepare_chat_access` can optionally wait for the instance first, then call `GetClawInstanceChatToken`.


In [ ]:
chat_access = client.workflows.prepare_chat_access(
    space_id=SPACE_ID,
    instance_id=INSTANCE_ID,
    wait=False,
)
print_json("PrepareChatAccess", chat_access)


## 8. Send a Message Through a WebSocket Session

Use `create_message_session` when you want the SDK to handle chat-token retrieval, connection setup, and receive timeout behavior for an ArkClaw instance.


In [ ]:
message = "Hello from arkclaw-python-sdk notebook. Please reply with one short sentence."

with client.create_message_session(
    space_id=SPACE_ID,
    instance_id=INSTANCE_ID,
    wait=True,
    wait_timeout=600,
    receive_timeout=120,
) as session:
    message_result = session.send_message(message, receive=True)

print_json("SendMessage", message_result)


## 9. Update Instance Model

Model updates should be performed after the instance is running. Replace the placeholders with model values supported by your environment.


In [ ]:
# Uncomment and fill real model values before running.
# update_model = client.instances.update_model(
#     instance_id=INSTANCE_ID,
#     model_name="replace-with-model-name",
#     model_source="replace-with-model-source",
#     runtime_options=runtime,
# )
# print_json("UpdateClawInstanceModel", update_model)


## 10. Bind an IM Channel

Only run this cell after you have real IM credentials. Keep secrets outside notebooks whenever possible.


In [ ]:
# Uncomment and fill real IM credentials before running.
# update_channel = client.instances.update_channel(
#     instance_id=INSTANCE_ID,
#     im_type="Feishu",
#     im_client_id="replace-with-im-client-id",
#     im_client_secret="replace-with-im-client-secret",
#     runtime_options=runtime,
# )
# print_json("UpdateClawInstanceChannel", update_channel)


## 11. Create and Inspect a Command Job

Command jobs are useful for operational checks across one or more ArkClaw instances. Start with a harmless command such as `echo`.


In [ ]:
job = client.workflows.create_command_job_and_wait(
    space_id=SPACE_ID,
    job_name=COMMAND_JOB_NAME,
    command_content="#!/bin/bash\necho hello-from-arkclaw-sdk",
    instance_ids=[INSTANCE_ID],
    timeout=600,
    interval=5,
)
print_json("CreateCommandJobAndWait", job)

job_id = job.get("job_id")
if job_id:
    job_log = client.command_jobs.get_log(
        space_id=SPACE_ID,
        job_id=job_id,
        instance_id=INSTANCE_ID,
        runtime_options=runtime,
    )
    print_json("GetCommandJobLog", job_log)


## 12. Error Handling Pattern

Catch `ValidationError` for local input/configuration problems and `ApiError` for service responses or transport failures normalized by the SDK.


In [ ]:
try:
    client.instances.get(space_id=SPACE_ID, instance_id=INSTANCE_ID, runtime_options=runtime)
except ValidationError as exc:
    print("Invalid local request:", exc)
except ApiError as exc:
    print("ArkClaw API error:", exc)
    print("request_id=", exc.request_id)
    print("code=", exc.code)
    print("retryable=", exc.retryable)
else:
    print("Instance query succeeded.")
